# Master Snakemake Pipeline
This section contains the fully polished, end-to-end Snakemake workflow. You can run these cells from top to bottom to execute the entire process cleanly.

In [1]:
# 1. Setup, Dependencies, and Configuration
import sys
import os
import yaml
import glob

# Install Snakemake if not present
!{sys.executable} -m pip install snakemake -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Define Core Paths
ROOT_DIR = '/content'
PROJECT_DIR = '/content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model'
modules_dir = os.path.join(PROJECT_DIR, 'modules')
python_scripts_dir = os.path.join(modules_dir, 'python_files')

os.makedirs(python_scripts_dir, exist_ok=True)
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# Convert Input Excel to YAML for Snakemake
excel_path = os.path.join(PROJECT_DIR, 'input_file.xlsx')
yaml_path = os.path.join(PROJECT_DIR, 'input_file.yaml')

if os.path.exists(excel_path):
    try:
        from modules.getting_input_data import get_input_data
        input_data = get_input_data(PROJECT_DIR, "input_file.xlsx")
        # Ensure we run all required scenarios and years
        input_data['scenario'] = ['GA', 'DE']
        input_data['year'] = ['2040', '2050']

        with open(yaml_path, 'w') as f:
            yaml.dump(input_data, f, default_flow_style=False)
        print("✅ Configuration loaded and saved to input_file.yaml")
    except Exception as e:
        print(f"❌ Error reading Excel: {e}")
else:
    print(f"❌ Missing {excel_path}! Please upload it.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 9.2 MB/s eta 0:00:00
Mounted at /content/drive
File found at: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/input_file.xlsx
The input data has been imported.
✅ Configuration loaded and saved to input_file.yaml


In [2]:
# 2. Convert Notebooks and Patch Scripts
print("--- 2a. Converting Notebooks ---")
nbs_to_convert = [
    "Reading_Demand_profiles.ipynb", "Reading_Gas_Infrastructure.ipynb",
    "Reading_Hydrogen.ipynb", "Reading_PEMMDB.ipynb", "Reading_RES_profiles.ipynb"
]
for nb_name in nbs_to_convert:
    nb_path = os.path.join(modules_dir, nb_name)
    if os.path.exists(nb_path):
        os.system(f"jupyter nbconvert --to python '{nb_path}' --output-dir='{python_scripts_dir}'")

print("--- 2b. Patching Scripts ---")
injection_code = f"""
# --- INJECTED BY PIPELINE ---
import yaml
import argparse
import sys
import os

parser = argparse.ArgumentParser()
parser.add_argument('--scenario', type=str, default='GA')
parser.add_argument('--year', type=str, default='2040')
args, unknown = parser.parse_known_args()

ROOT_DIR = '/content'
PROJECT_DIR = '{PROJECT_DIR}'

yaml_path = os.path.join(PROJECT_DIR, 'input_file.yaml')
with open(yaml_path, 'r') as f:
    input_data = yaml.safe_load(f)

input_data['scenario'] = args.scenario
input_data['year'] = int(args.year)  # Cast to integer for Pandas indexing

DATA_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]), input_data["raw_data_dir"])
output_dir = os.path.join(PROJECT_DIR, str(input_data["data_set"]), input_data['inter_dir'], input_data["project_name"], input_data["scenario"], str(input_data["year"]))
os.makedirs(output_dir, exist_ok=True)
# --------------------------\n
"""

import re
for py_file in glob.glob(os.path.join(python_scripts_dir, '*.py')):
    if 'getting_input_data' in py_file: continue

    with open(py_file, 'r') as f:
        lines = f.readlines()

    with open(py_file, 'w') as f:
        f.write(injection_code)
        for line in lines:
            if 'getting_input_data' in line or 'input_data = get_input_data' in line:
                f.write('# [BLOCKED EXCEL] ' + line)
            elif any(x in line for x in ['ROOT_DIR =', 'PROJECT_DIR =', 'DATA_DIR =', 'output_dir =']) and not line.strip().startswith('#'):
                f.write('# [BLOCKED PATH] ' + line)
            elif any(x in line for x in ['drive.mount', 'google.colab', 'get_ipython', '!pip', '%matplotlib']):
                f.write('# [REMOVED COLAB] ' + line)
            else:
                line = line.replace('display(', 'print(')

                # More robust regex replacement for target_year casting
                line = re.sub(r'\[\s*target_year\s*\]', '[int(target_year)]', line)
                line = re.sub(r'\[\[\s*target_year\s*\]\]', '[[int(target_year)]]', line)
                line = re.sub(r',\s*target_year\s*\]', ', int(target_year)]', line)

                f.write(line)

print("✅ All Python scripts successfully converted and patched for Snakemake.")

--- 2a. Converting Notebooks ---
--- 2b. Patching Scripts ---
✅ All Python scripts successfully converted and patched for Snakemake.


In [3]:
input_data

{'project_name': 'Europe',
 'regions': '["Albania", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "United Kingdom"]',
 'zones': '["AL00", "AT00", "BA00", "BE00", "BG00", "CH00", "CY00", "CZ00", "DE00", "DKE1", "DKW1", "EE00", "ES00", "FI00", "FR00", "GR00", "GR03", "HR00", "HU00", "IE00", "ITCA", "ITCN", "ITCS", "ITN1", "ITS1", "ITSA", "ITSI", "LT00", "LUB1", "LUF1", "LUG1", "LUV1", "LV00", "ME00", "SE02", "SE03", "SE04", "SI00", "SK00", "UK00", "UKNI", "MK00", "MT00", "NL00", "NOM1", "NON1", "NOS0", "PL00", "PT00", "RO00", "RS00", "SE01", "SE02", "SE03", "SE04", "SI00", "SK00", "UK00", "UKNI"]',
 'data_set': 'TYNDP_scenario_2024',
 'year': ['204

In [4]:
# 3. Generate Snakefile, Run Pipeline, and Verify
print("--- 3a. Generating Snakefile ---")
snakefile_content = (
    "configfile: 'input_file.yaml'\n\n"
    "rule all:\n"
    "    input:\n"
    "        expand(\"{data_set}/{inter_dir}/{project_name}/{scenario}/{year}/RES_profiles.csv\",\n"
    "               data_set=config['data_set'],\n"
    "               inter_dir=config['inter_dir'],\n"
    "               scenario=config['scenario'],\n"
    "               year=config['year'],\n"
    "               project_name=config['project_name'])\n\n"
    "rule run_pipeline:\n"
    "    input:\n"
    f"        \"{input_data['input_data_path']}\"\n"
    "    output:\n"
    f"        \"{input_data['data_set']}/{input_data['inter_dir']}/{input_data['project_name']}/{{scenario}}/{{year}}/RES_profiles.csv\"\n"
    "    shell:\n"
    "        \"\"\"\n"
    f"        python {python_scripts_dir}/Reading_PEMMDB.py --scenario {{wildcards.scenario}} --year {{wildcards.year}}\n"
    f"        python {python_scripts_dir}/Reading_Demand_profiles.py --scenario {{wildcards.scenario}} --year {{wildcards.year}}\n"
    f"        python {python_scripts_dir}/Reading_Gas_Infrastructure.py --scenario {{wildcards.scenario}} --year {{wildcards.year}}\n"
    f"        python {python_scripts_dir}/Reading_Hydrogen.py --scenario {{wildcards.scenario}} --year {{wildcards.year}}\n"
    f"        python {python_scripts_dir}/Reading_RES_profiles.py --scenario {{wildcards.scenario}} --year {{wildcards.year}}\n"
    "        \"\"\"\n"
)

with open(os.path.join(PROJECT_DIR, 'Snakefile'), 'w') as f:
    f.write(snakefile_content)

print("\n--- 3b. Running Snakemake Pipeline ---")
os.chdir(PROJECT_DIR)
# Force run all combinations
!snakemake -F --cores 1

print("\n--- 3c. Verifying Final Outputs ---")
# Fixed: dynamically use project_name instead of hardcoding 'Italy'
project_name = input_data['project_name']
output_pattern = os.path.join(PROJECT_DIR, f"2024/intermediate_data/{project_name}/*/*/RES_profiles.csv")
generated_files = glob.glob(output_pattern)

if len(generated_files) == 4:
    print(f"🎉 SUCCESS! Found all {len(generated_files)} expected output files for {project_name}:")
else:
    print(f"⚠️ Partial/Failed run. Found {len(generated_files)} output files (expected 4) for {project_name}:")

for f in sorted(generated_files):
    print(f" - {f}")

--- 3a. Generating Snakefile ---

--- 3b. Running Snakemake Pipeline ---
Assuming unrestricted shared filesystem usage.

SNAKEMAKE
  Date: 2026-06-14 08:11:11
  Workflow ID: b02575bb-07db-470e-8c82-3d049ff1debb
  Platform: Linux-6.6.122+-x86_64-with-glibc2.35
  Host: b9ab3f491480
  User: root
  Snakemake version: 9.23.0
  Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
  Command: /usr/local/bin/snakemake -F --cores 1
  Snakefile: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/Snakefile
  Base directory: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model
  Run directory: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model
  Working directory: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model
  Config file(s): ['input_file.yaml']
  Config MD5: f09c8b16ab122ce341827827b0c6590e

Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 1 (use --cores to define parallelism)
Rules claiming more threads will be scaled down.
Job stats:
jo

# REST  


In [5]:
nodes = [
    'AL00', 'AT00', 'BA00', 'BE00', 'BG00', 'CH00', 'CY00', 'CZ00', 'DE00',
    'DKE1', 'DKW1', 'EE00', 'ES00', 'FI00', 'FR00', 'GR00', 'GR03', 'HR00',
    'HU00', 'IE00', 'ITCA', 'ITCN', 'ITCS', 'ITN1', 'ITS1', 'ITSA', 'ITSI',
    'LT00', 'LUB1', 'LUF1', 'LUG1', 'LUV1', 'LV00', 'ME00', 'MK00', 'MT00',
    'NL00', 'NOM1', 'NON1', 'NOS0', 'PL00', 'PT00', 'RO00', 'RS00', 'SE01',
    'SE02', 'SE03', 'SE04', 'SI00', 'SK00', 'UK00', 'UKNI'
]

print(f"Created a list with {len(nodes)} nodes.")

Created a list with 52 nodes.


In [6]:
country_mapping = {
    'AL': 'Albania', 'AT': 'Austria', 'BA': 'Bosnia and Herzegovina',
    'BE': 'Belgium', 'BG': 'Bulgaria', 'CH': 'Switzerland',
    'CY': 'Cyprus', 'CZ': 'Czechia', 'DE': 'Germany',
    'DK': 'Denmark', 'EE': 'Estonia', 'ES': 'Spain',
    'FI': 'Finland', 'FR': 'France', 'GR': 'Greece',
    'HR': 'Croatia', 'HU': 'Hungary', 'IE': 'Ireland',
    'IT': 'Italy', 'LT': 'Lithuania', 'LU': 'Luxembourg',
    'LV': 'Latvia', 'ME': 'Montenegro', 'MK': 'North Macedonia',
    'MT': 'Malta', 'NL': 'Netherlands', 'NO': 'Norway',
    'PL': 'Poland', 'PT': 'Portugal', 'RO': 'Romania',
    'RS': 'Serbia', 'SE': 'Sweden', 'SI': 'Slovenia',
    'SK': 'Slovakia', 'UK': 'United Kingdom'
}

# Extract the unique countries based on the first two letters of each node
countries = sorted(list(set(country_mapping[node[:2]] for node in nodes if node[:2] in country_mapping)))

print("Countries involved:")
for country in countries:
    print(f"- {country}")

Countries involved:
- Albania
- Austria
- Belgium
- Bosnia and Herzegovina
- Bulgaria
- Croatia
- Cyprus
- Czechia
- Denmark
- Estonia
- Finland
- France
- Germany
- Greece
- Hungary
- Ireland
- Italy
- Latvia
- Lithuania
- Luxembourg
- Malta
- Montenegro
- Netherlands
- North Macedonia
- Norway
- Poland
- Portugal
- Romania
- Serbia
- Slovakia
- Slovenia
- Spain
- Sweden
- Switzerland
- United Kingdom
